# Deployments

Ein Deployment verwaltet über ReplicaSets die gewünschte Anzahl gleichartiger Pods.

Notebook mit Python-Kernel aus diesem Verzeichnis ausführen. Die Ressourcen hello-deploy und lb-svc sowie das Label app=hello-world müssen für diese Übung frei sein.

Voraussetzung: Das Image javacream/hostinfo:1.0-lb muss für alle Kubernetes-Nodes verfügbar sein. Der Quellcode liegt im Unterverzeichnis image. Bei Verwendung von Docker Hub das Image dort vorab veröffentlichen (docker compose build und docker compose push im Verzeichnis image). Ein lokaler Docker-Build allein stellt das Image nicht im K3s-Containerd bereit.

## 1. Deployment untersuchen
Drei Replikate, Pod-Template, Selector und Ressourcenlimits. Die Update-Strategie wird später behandelt.
Das Image javacream/hostinfo:1.0-lb liefert über NGINX auf Port 80 eine HTML-Seite mit Hostname und Systeminformationen. Der Hostname entspricht standardmäßig dem Pod-Namen.

In [ ]:
%%bash
set -e
cat deploy.yml
kubectl apply -f deploy.yml
kubectl wait --for=jsonpath='{.status.availableReplicas}'=3 deployment/hello-deploy --timeout=360s

## 2. Deployment, ReplicaSet und Pods
Das Deployment verwaltet ReplicaSets; diese verwalten die Pods. Die Namen der Pods werden automatisch erzeugt.

In [ ]:
%%bash
set -e
kubectl get deployments,replicasets,pods -l app=hello-world -o wide
kubectl get deployment hello-deploy
kubectl get replicasets -l app=hello-world
kubectl get pods -l app=hello-world -o custom-columns='NAME:.metadata.name,OWNER:.metadata.ownerReferences[0].name,NODE:.spec.nodeName'

## 3. Self-Healing
Einen Pod löschen. Das ReplicaSet erzeugt einen Ersatz, um die gewünschte Anzahl wiederherzustellen. Die Pod-Namen vor und nach dem Löschen vergleichen.

In [ ]:
%%bash
set -e
kubectl get pods -l app=hello-world
POD=$(kubectl get pods -l app=hello-world -o jsonpath='{.items[0].metadata.name}')
echo "Lösche $POD"
kubectl delete pod "$POD"
kubectl get pods -l app=hello-world
kubectl wait --for=jsonpath='{.status.availableReplicas}'=3 deployment/hello-deploy --timeout=360s
kubectl get pods -l app=hello-world

## 4. Service und Verteilung der Anfragen

Der Service lb-svc wählt die Pods anhand des Labels app=hello-world aus. K3s ServiceLB stellt den LoadBalancer-Service auf verfügbaren Nodes über Port 8080 bereit. Der Service leitet an targetPort 80 des Containers weiter. Die Node-IP ist nur der Einstiegspunkt in den Cluster, nicht der Node, auf dem die Anfrage verarbeitet wird.

Standardmäßig gilt externalTrafficPolicy: Cluster. Anfragen können daher an alle bereiten, ausgewählten Pods weitergeleitet werden, auch auf anderen Nodes.

```text
Host → Node-IP:8080 → Service → Pod A (Node 1)
                             → Pod B (Node 2)
                             → Pod C (Node 3)
```

Die tatsächliche Verteilung der Pods auf Nodes hängt vom Scheduler ab.

In [ ]:
%%bash
set -e
cat lb.yml
kubectl apply -f lb.yml
kubectl get svc lb-svc
kubectl get nodes -o wide
kubectl get pods -l app=hello-world -o wide

Die unter EXTERNAL-IP angezeigte IP muss vom Ubuntu-Host erreichbar sein. Bei unserer Docker-basierten K3s-Umgebung ist dies normalerweise eine Node-IP im Docker-Netzwerk. Falls EXTERNAL-IP noch pending ist, die Service-Ausgabe erneut prüfen. Im Browser kann dieselbe IP mit Port 8080 aufgerufen werden.

Zehn unabhängige HTTP-Verbindungen an dieselbe IP senden. Aus jeder HTML-Antwort wird der Hostname extrahiert und mit den Pod-Namen und Nodes aus kubectl get pods -o wide verglichen. Die Linux-Distribution beschreibt das Container-Image, nicht den Node. Die Verteilung erfolgt nicht zwingend nach Round Robin.

In [ ]:
%%bash
set -eo pipefail
NODE_IP=$(kubectl get svc lb-svc -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
if [ -z "$NODE_IP" ]; then
    echo "Keine LoadBalancer-IP verfügbar. Service und ServiceLB prüfen."
    exit 1
fi
echo "Ziel: http://$NODE_IP:8080/"
for i in {1..10}; do
    echo "Aufruf $i"
    curl -fsS --connect-timeout 5 --max-time 10 "http://$NODE_IP:8080/" | sed -n 's/.*<dt>Hostname<\/dt><dd>\([^<]*\)<\/dd>.*/\1/p'
done

### Load Balancing im Browser

Ein Kubernetes-TCP-Service verteilt TCP-Verbindungen, nicht einzelne HTTP-Requests. Browser können mehrere HTTP-Requests über dieselbe TCP-Verbindung senden (HTTP Keep-Alive). Außerdem können sie HTML-Antworten aus dem Cache verwenden oder mit bedingten Requests validieren. Dadurch ist die Lastverteilung beim normalen Reload nicht unbedingt sichtbar.

Das Image `javacream/hostinfo:1.0-lb` ist bewusst für diese Demonstration konfiguriert:

- `keepalive_timeout 0;`: NGINX schließt die HTTP/1.x-Verbindung nach der Antwort.
- `add_header Cache-Control "no-store" always;`: NGINX weist den Browser an, die Antwort nicht zu speichern.

Die Seite unter der zuvor ermittelten External-IP auf Port 8080 im Browser öffnen und mehrfach neu laden. Bei bereits gecachten Antworten einmal Shift + Reload verwenden. Auch bei neuen Verbindungen ist ein Wechsel des Pods nicht bei jedem Aufruf garantiert. Diese Einstellungen dienen der Demonstration und sind keine allgemeine Empfehlung für produktive Webserver.

### Einfluss von externalTrafficPolicy

| Einstellung | Weiterleitung |
|---|---|
| Cluster (Default) | An bereite Pods auf beliebigen Nodes |
| Local | Nur an bereite Pods auf dem angesprochenen Node; ohne lokale Ziele ist dort kein Zugriff möglich |

In dieser Übung bleibt die Standardeinstellung Cluster unverändert. Die Anwendung ist von der Platzierung der Pods entkoppelt. Port-Forwarding auf einen Service wäre für diese Demonstration ungeeignet, da es einen einzelnen Pod auswählt. Weitere Service-Konzepte folgen im nächsten Kapitel.

## 5. Skalierung
Die gewünschte Anzahl zunächst auf fünf, anschließend auf zwei setzen. Das Pod-Template bleibt unverändert.
Der Service bleibt unverändert. Nach jeder Skalierung erneut dieselbe Service-Adresse aufrufen und die möglichen Ziel-Pods vergleichen.

In [ ]:
%%bash
set -e
kubectl scale deployment hello-deploy --replicas=5
kubectl wait --for=jsonpath='{.status.availableReplicas}'=5 deployment/hello-deploy --timeout=360s
kubectl get deployment hello-deploy
kubectl get pods -l app=hello-world -o wide

In [ ]:
%%bash
set -eo pipefail
NODE_IP=$(kubectl get svc lb-svc -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
if [ -z "$NODE_IP" ]; then
    echo "Keine LoadBalancer-IP verfügbar. Service und ServiceLB prüfen."
    exit 1
fi
echo "Ziel: http://$NODE_IP:8080/"
for i in {1..10}; do
    echo "Aufruf $i"
    curl -fsS --connect-timeout 5 --max-time 10 "http://$NODE_IP:8080/" | sed -n 's/.*<dt>Hostname<\/dt><dd>\([^<]*\)<\/dd>.*/\1/p'
done

In [ ]:
%%bash
set -e
kubectl scale deployment hello-deploy --replicas=2
kubectl wait --for=jsonpath='{.status.availableReplicas}'=2 deployment/hello-deploy --timeout=360s
kubectl get pods -l app=hello-world -o wide

In [ ]:
%%bash
set -eo pipefail
NODE_IP=$(kubectl get svc lb-svc -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
if [ -z "$NODE_IP" ]; then
    echo "Keine LoadBalancer-IP verfügbar. Service und ServiceLB prüfen."
    exit 1
fi
echo "Ziel: http://$NODE_IP:8080/"
for i in {1..10}; do
    echo "Aufruf $i"
    curl -fsS --connect-timeout 5 --max-time 10 "http://$NODE_IP:8080/" | sed -n 's/.*<dt>Hostname<\/dt><dd>\([^<]*\)<\/dd>.*/\1/p'
done

## 6. Aufräumen
Beim Löschen des Deployments werden die zugehörigen ReplicaSets und Pods standardmäßig mitgelöscht. Der unabhängig angelegte Service muss separat gelöscht werden.

In [ ]:
%%bash
set -e
kubectl delete -f deploy.yml
kubectl get deployments,replicasets,pods -l app=hello-world
kubectl get service lb-svc
kubectl delete -f lb.yml

## Fazit
Ein Deployment beschreibt den gewünschten Zustand einer Anwendung, insbesondere die Anzahl der Replikate und das Pod-Template. ReplicaSets sorgen dafür, dass die gewünschte Anzahl an Pods bereitgestellt wird. Ein Service bietet einen stabilen Zugang zu den wechselnden Pods, unabhängig von deren Platzierung auf Nodes.